# EVRP-TW-D — Full Dataset Analysis + Easiest Unanchored Cus_100 Ranking

Read-only analysis of `dataset/EVRPTWD/` (EVRP-TW-D-B v1.0).

## Goals
1. Full inventory of anchored / unanchored across all scales
2. Deep focus on **Cus_100** (the scale used for Transformer MARL training)
3. Rank **all unanchored Cus_100** instances from **easiest → hardest** using physics proxies (no exact solver)
4. Relate findings to the project split: RC-only, 800 train / 200 test

## Ease score (higher = easier)
Composite of:
- wider time windows / more slack
- lower demand vs capacity (fewer min vehicles)
- lower depot→customer energy vs battery
- more compact geography
- more charging stations
- family prior: C ≳ R ≳ RC (typical literature difficulty)
- TW regime: wide ≫ narrow

In [ ]:
from __future__ import annotations

import json
import math
import re
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

plt.style.use("ggplot")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 60)

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "EVRPTWD":
    cand = NOTEBOOK_DIR / "dataset" / "EVRPTWD"
    if cand.exists():
        NOTEBOOK_DIR = cand

ROOT = NOTEBOOK_DIR
OUT_DIR = ROOT / "analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert (ROOT / "anchored").is_dir() and (ROOT / "unanchored").is_dir(), ROOT
print("EVRPTWD root:", ROOT)
print("Analysis outputs:", OUT_DIR)

## 1. Global inventory (all scales)

In [ ]:
NAME_RE = re.compile(
    r"solomon_dataset_(?P<id>\d+)_(?P<family>R|C|RC)_(?P<tw>wide|narrow)_(?P<ts>\d+)\.txt$"
)

def inventory_all() -> pd.DataFrame:
    rows = []
    for split in ["anchored", "unanchored"]:
        for scale_dir in sorted((ROOT / split).glob("Cus_*")):
            scale = int(scale_dir.name.replace("Cus_", ""))
            sol = scale_dir / "solomon"
            if not sol.is_dir():
                continue
            for path in sol.glob("*.txt"):
                m = NAME_RE.match(path.name)
                rows.append({
                    "split": split,
                    "scale": scale,
                    "family": m.group("family") if m else "?",
                    "tw": m.group("tw") if m else "?",
                    "instance_id": int(m.group("id")) if m else -1,
                    "stem": path.stem,
                    "path": str(path),
                })
    return pd.DataFrame(rows)

inv = inventory_all()
print("Total Solomon instances:", len(inv))
display(inv.groupby(["split", "scale"]).size().unstack("scale").fillna(0).astype(int))
display(Markdown("### Family mix"))
display(inv.groupby(["split", "scale", "family"]).size().unstack("family").fillna(0).astype(int))
display(Markdown("### TW mix"))
display(inv.groupby(["split", "scale", "tw"]).size().unstack("tw").fillna(0).astype(int))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, split in zip(axes, ["anchored", "unanchored"]):
    sub = inv[inv["split"] == split]
    ct = sub.groupby(["scale", "family"]).size().unstack("family").fillna(0)
    ct.plot(kind="bar", stacked=True, ax=ax, title=f"{split}: count by scale × family")
    ax.set_xlabel("# customers")
    ax.set_ylabel("# instances")
plt.tight_layout()
plt.show()

## 2. Solomon parser (nodes + vehicle params + footer meta)

In [ ]:
PARAM_RE = {
    "Q": re.compile(r"Q\s+Vehicle fuel tank capacity\s+/([\d.+-eE]+)/"),
    "C": re.compile(r"C\s+Vehicle load capacity\s+/([\d.+-eE]+)/"),
    "r": re.compile(r"r\s+fuel consumption rate\s+/([\d.+-eE]+)/"),
    "g": re.compile(r"g\s+inverse refueling rate\s+/([\d.+-eE]+)/"),
    "v": re.compile(r"v\s+average Velocity\s+/([\d.+-eE]+)/"),
    "gv": re.compile(r"gv\s+charging speed\s+/([\d.+-eE]+)/"),
}


def parse_solomon(path: Path) -> dict:
    text = path.read_text(encoding="utf-8", errors="replace")
    lines = [ln.rstrip() for ln in text.splitlines() if ln.strip()]
    nodes = []
    params: dict[str, float] = {}
    meta: dict[str, str] = {}
    in_footer = False
    for ln in lines:
        if ln.startswith("StringID"):
            continue
        if ln.startswith("-"):
            in_footer = True
            continue
        matched = False
        for k, pat in PARAM_RE.items():
            m = pat.search(ln)
            if m:
                params[k] = float(m.group(1))
                matched = True
                break
        if matched:
            continue
        if in_footer:
            if "/" in ln:
                key, _, rest = ln.partition("/")
                val = rest.rsplit("/", 1)[0].strip() if "/" in rest else rest.strip()
                meta[key.strip()] = val
            continue
        parts = ln.split()
        if len(parts) >= 8 and parts[1] in {"d", "f", "c"}:
            nodes.append({
                "id": parts[0],
                "type": parts[1],
                "x": float(parts[2]),
                "y": float(parts[3]),
                "demand": float(parts[4]),
                "ready": float(parts[5]),
                "due": float(parts[6]),
                "service": float(parts[7]),
            })
    df = pd.DataFrame(nodes)
    return {"path": path, "nodes": df, "params": params, "meta": meta}


sample = parse_solomon(next((ROOT / "unanchored" / "Cus_100" / "solomon").glob("*.txt")))
print("Sample:", sample["path"].name)
print("params:", sample["params"])
print("meta keys:", list(sample["meta"].keys())[:8], "...")
display(sample["nodes"].groupby("type").size().rename("count").to_frame())
display(sample["nodes"].head(8))

## 3. Feature extraction for difficulty / ease

For each instance we compute structural proxies used later for ranking.

In [ ]:
FAMILY_EASE = {"C": 1.0, "R": 0.55, "RC": 0.25}  # literature-style prior
TW_EASE = {"wide": 1.0, "narrow": 0.35}


def instance_features(path: Path, *, split: str, scale: int) -> dict:
    parsed = parse_solomon(path)
    df = parsed["nodes"]
    p = parsed["params"]
    m = NAME_RE.match(path.name)
    family = m.group("family") if m else parsed["meta"].get("instance_type", "?")
    tw = m.group("tw") if m else "?"

    cust = df[df["type"] == "c"]
    st = df[df["type"] == "f"]
    dep = df[df["type"] == "d"].iloc[0]

    Q = float(p["Q"]); Cload = float(p["C"]); r = float(p["r"]); v = float(p["v"])
    horizon = float(dep["due"]) if float(dep["due"]) > 0 else float(cust["due"].max())
    demand_sum = float(cust["demand"].sum())
    m_min = max(1, int(math.ceil(demand_sum / Cload))) if Cload > 0 else 1

    dx = cust["x"].to_numpy() - float(dep["x"])
    dy = cust["y"].to_numpy() - float(dep["y"])
    dist_dep = np.sqrt(dx * dx + dy * dy)
    energy_dep = dist_dep * r
    tw_width = (cust["due"] - cust["ready"]).to_numpy()
    # rough earliest-arrival slack using depot→customer travel time
    travel = dist_dep / max(v, 1e-9)
    arrival = travel  # from t=0 at depot as proxy
    service_start = np.maximum(arrival, cust["ready"].to_numpy())
    slack = cust["due"].to_numpy() - service_start

    xs = cust["x"].to_numpy(); ys = cust["y"].to_numpy()
    spat_std = float(np.sqrt(np.var(xs) + np.var(ys)))

    return {
        "split": split,
        "scale": scale,
        "stem": path.stem,
        "family": family,
        "tw": tw,
        "n_customer": int(len(cust)),
        "n_station": int(len(st)),
        "Q": Q,
        "C_load": Cload,
        "r": r,
        "v": v,
        "g": float(p.get("g", np.nan)),
        "horizon": horizon,
        "demand_sum": demand_sum,
        "demand_mean": float(cust["demand"].mean()),
        "m_min": m_min,
        "load_pressure": demand_sum / max(Cload, 1e-9),
        "tw_width_mean": float(np.mean(tw_width)),
        "tw_width_min": float(np.min(tw_width)),
        "tw_width_frac": float(np.mean(tw_width) / max(horizon, 1e-9)),
        "slack_mean": float(np.mean(slack)),
        "slack_frac_positive": float(np.mean(slack > 0)),
        "tight_tw_frac": float(np.mean(tw_width < 0.15 * horizon)),
        "service_mean": float(cust["service"].mean()),
        "dist_depot_mean": float(np.mean(dist_dep)),
        "dist_depot_max": float(np.max(dist_dep)),
        "energy_depot_mean": float(np.mean(energy_dep)),
        "energy_depot_max": float(np.max(energy_dep)),
        "batt_pressure": float(np.mean(energy_dep) / max(Q, 1e-9)),
        "batt_pressure_max": float(np.max(energy_dep) / max(Q, 1e-9)),
        "spat_std": spat_std,
        "station_per_cust": float(len(st) / max(len(cust), 1)),
    }


def ease_score(row: pd.Series) -> float:
    """Higher = easier. All terms roughly in [0,1] then weighted."""
    tw = float(np.clip(row["tw_width_frac"] / 0.35, 0, 1))  # ~0.35+ is very wide
    slack = float(np.clip(row["slack_mean"] / max(row["horizon"], 1e-9) / 0.25, 0, 1))
    load = float(np.clip(1.0 - (row["load_pressure"] - 1.0) / 8.0, 0, 1))  # m_min-ish
    batt = float(np.clip(1.0 - row["batt_pressure"] / 0.35, 0, 1))
    compact = float(np.clip(1.0 - row["spat_std"] / 45.0, 0, 1))
    stations = float(np.clip(row["station_per_cust"] / 0.25, 0, 1))
    tight_pen = float(np.clip(1.0 - row["tight_tw_frac"], 0, 1))
    fam = FAMILY_EASE.get(row["family"], 0.4)
    twreg = TW_EASE.get(row["tw"], 0.5)
    return float(
        0.18 * tw
        + 0.14 * slack
        + 0.12 * load
        + 0.12 * batt
        + 0.10 * compact
        + 0.08 * stations
        + 0.08 * tight_pen
        + 0.10 * fam
        + 0.08 * twreg
    )

## 4. Deep Cus_100 analysis — scan **all unanchored** (10,000) + all anchored (100)

This is the heavy cell. Results are cached under `analysis/`.

In [ ]:
CACHE_UN = OUT_DIR / "cus100_unanchored_features.parquet"
CACHE_AN = OUT_DIR / "cus100_anchored_features.parquet"
RANK_CSV = OUT_DIR / "cus100_unanchored_easiest_to_hardest.csv"


def scan_cus100(split: str, cache_path: Path, force: bool = False) -> pd.DataFrame:
    if cache_path.is_file() and not force:
        df = pd.read_parquet(cache_path)
        print(f"Loaded cache {cache_path.name}: {len(df)} rows")
        return df
    sol = ROOT / split / "Cus_100" / "solomon"
    paths = sorted(sol.glob("*.txt"))
    print(f"Scanning {split} Cus_100: {len(paths)} files ...")
    t0 = time.time()
    rows = []
    for i, path in enumerate(paths):
        rows.append(instance_features(path, split=split, scale=100))
        if (i + 1) % 1000 == 0 or (i + 1) == len(paths):
            print(f"  {i+1}/{len(paths)}  ({time.time()-t0:.1f}s)")
    df = pd.DataFrame(rows)
    df["ease_score"] = df.apply(ease_score, axis=1)
    df["ease_rank"] = df["ease_score"].rank(ascending=False, method="first").astype(int)
    df = df.sort_values(["ease_score", "stem"], ascending=[False, True]).reset_index(drop=True)
    df.to_parquet(cache_path, index=False)
    print(f"Wrote {cache_path}")
    return df


feat_un = scan_cus100("unanchored", CACHE_UN, force=False)
feat_an = scan_cus100("anchored", CACHE_AN, force=False)

display(Markdown("### Unanchored Cus_100 — summary by family × TW"))
display(
    feat_un.groupby(["family", "tw"]).agg(
        n=("stem", "count"),
        ease_mean=("ease_score", "mean"),
        tw_width_frac=("tw_width_frac", "mean"),
        load_pressure=("load_pressure", "mean"),
        batt_pressure=("batt_pressure", "mean"),
        spat_std=("spat_std", "mean"),
        m_min=("m_min", "mean"),
        n_station=("n_station", "mean"),
    ).round(3)
)

## 5. Cus_100 distributions (unanchored)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
plot_cols = [
    ("ease_score", "Ease score (↑ easier)"),
    ("tw_width_frac", "Mean TW width / horizon"),
    ("load_pressure", "Total demand / capacity"),
    ("batt_pressure", "Mean depot→cust energy / Q"),
    ("spat_std", "Spatial std (compactness↓)"),
    ("m_min", "Min vehicles (ceil demand/C)"),
]
for ax, (col, title) in zip(axes.ravel(), plot_cols):
    for fam, g in feat_un.groupby("family"):
        ax.hist(g[col], bins=40, alpha=0.45, label=fam, density=True)
    ax.set_title(title)
    ax.legend(fontsize=8)
plt.suptitle("Unanchored Cus_100 feature distributions by family", y=1.02)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for tw, g in feat_un.groupby("tw"):
    ax.hist(g["ease_score"], bins=40, alpha=0.5, label=tw, density=True)
ax.set_title("Ease score: wide vs narrow (unanchored Cus_100)")
ax.set_xlabel("ease_score")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Easiest → hardest ranking (UNANCHORED Cus_100 only)

Full ordered table is saved to CSV. Below: **top 50 easiest** and **top 50 hardest**.

In [ ]:
rank = feat_un.sort_values(["ease_score", "stem"], ascending=[False, True]).reset_index(drop=True)
rank.insert(0, "rank_easiest", np.arange(1, len(rank) + 1))
cols_show = [
    "rank_easiest", "stem", "family", "tw", "ease_score",
    "tw_width_frac", "slack_mean", "load_pressure", "m_min",
    "batt_pressure", "spat_std", "n_station", "tight_tw_frac",
]
rank[cols_show].to_csv(RANK_CSV, index=False)
print("Wrote full ranking:", RANK_CSV)
print("N =", len(rank))

display(Markdown("### Top 50 easiest (unanchored Cus_100)"))
display(rank.loc[:49, cols_show].round(4))

display(Markdown("### Top 50 hardest (unanchored Cus_100)"))
display(rank.loc[rank.index[-50:], cols_show].round(4))

display(Markdown("### Easiest 20 stems (copy-friendly)"))
print("\n".join(rank.loc[:19, "stem"].tolist()))

In [ ]:
# Composition of easiest / hardest tails
easy_tail = rank.head(500)
hard_tail = rank.tail(500)
cmp = pd.DataFrame({
    "easiest_500": easy_tail.groupby(["family", "tw"]).size(),
    "hardest_500": hard_tail.groupby(["family", "tw"]).size(),
}).fillna(0).astype(int)
display(Markdown("### Family × TW composition of easiest vs hardest 500"))
display(cmp)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
easy_tail.groupby("family").size().plot(kind="bar", ax=axes[0], title="Easiest 500 — family")
hard_tail.groupby("family").size().plot(kind="bar", ax=axes[1], title="Hardest 500 — family", color="C1")
plt.tight_layout()
plt.show()

## 7. Spatial examples — easiest vs hardest

In [ ]:
def plot_instance(stem: str, ax, title: str):
    path = ROOT / "unanchored" / "Cus_100" / "solomon" / f"{stem}.txt"
    parsed = parse_solomon(path)
    df = parsed["nodes"]
    for typ, marker, label, s in [("d", "s", "depot", 60), ("f", "^", "station", 40), ("c", "o", "customer", 12)]:
        sub = df[df["type"] == typ]
        ax.scatter(sub["x"], sub["y"], marker=marker, s=s, label=label, alpha=0.85)
    ax.set_title(title, fontsize=9)
    ax.set_aspect("equal")


fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, stem in zip(axes[0], rank["stem"].head(3)):
    row = rank[rank["stem"] == stem].iloc[0]
    plot_instance(stem, ax, f"EASY #{int(row.rank_easiest)} {row.family}/{row.tw}\nease={row.ease_score:.3f}")
axes[0, 0].legend(fontsize=7, loc="best")
for ax, stem in zip(axes[1], rank["stem"].tail(3)):
    row = rank[rank["stem"] == stem].iloc[0]
    plot_instance(stem, ax, f"HARD #{int(row.rank_easiest)} {row.family}/{row.tw}\nease={row.ease_score:.3f}")
plt.suptitle("Unanchored Cus_100 — easiest vs hardest layouts", y=1.02)
plt.tight_layout()
plt.show()

## 8. RC-only view (matches current MARL train/test study)

Project uses **unanchored Cus_100 RC**, 800 train / 200 test.

In [ ]:
rc = rank[rank["family"] == "RC"].copy().reset_index(drop=True)
rc.insert(0, "rc_rank_easiest", np.arange(1, len(rc) + 1))
rc_csv = OUT_DIR / "cus100_unanchored_RC_easiest_to_hardest.csv"
rc_cols = ["rc_rank_easiest", "rank_easiest"] + cols_show[1:]
rc[rc_cols].to_csv(rc_csv, index=False)
print("RC instances:", len(rc))
print("Wrote", rc_csv)

display(Markdown("### Easiest 30 RC (unanchored Cus_100)"))
display(rc.loc[:29, rc_cols].round(4))

display(Markdown("### Hardest 30 RC (unanchored Cus_100)"))
display(rc.loc[rc.index[-30:], rc_cols].round(4))

# Overlap with project split if present
split_path = ROOT / "splits" / "cus100_rc_train800_test200_seed42.json"
if split_path.is_file():
    split = json.loads(split_path.read_text(encoding="utf-8"))
    train_set = set(split["train_instances"])
    test_set = set(split["test_instances"])
    rc["in_train"] = rc["stem"].isin(train_set)
    rc["in_test"] = rc["stem"].isin(test_set)
    display(Markdown("### Ease of current 800/200 RC split"))
    for label, mask in [("train800", rc["in_train"]), ("test200", rc["in_test"])]:
        sub = rc[mask]
        print(
            f"{label}: n={len(sub)}  ease_mean={sub.ease_score.mean():.4f}  "
            f"ease_median={sub.ease_score.median():.4f}  "
            f"wide={int((sub.tw=='wide').sum())} narrow={int((sub.tw=='narrow').sum())}"
        )
    display(Markdown("### Easiest 15 RC that are in the **train** split"))
    display(rc[rc["in_train"]].head(15)[rc_cols].round(4))
    display(Markdown("### Easiest 15 RC that are in the **test** split"))
    display(rc[rc["in_test"]].head(15)[rc_cols].round(4))
else:
    print("Project split JSON not found yet:", split_path)

## 9. Anchored Cus_100 vs unanchored (sanity)

Anchored is the fixed eval board (with Gurobi refs). Compare feature means only.

In [ ]:
cmp_means = pd.concat([
    feat_un.assign(source="unanchored"),
    feat_an.assign(source="anchored"),
], ignore_index=True)
display(
    cmp_means.groupby("source")[[
        "ease_score", "tw_width_frac", "load_pressure", "batt_pressure",
        "spat_std", "m_min", "n_station",
    ]].mean().round(4)
)

# Gurobi refs if present
gpath = ROOT / "anchored" / "Cus_100" / "gurobi_reference_cus100.csv"
if gpath.is_file():
    gref = pd.read_csv(gpath)
    # instance_id column is stem-like name without .txt
    gref = gref.rename(columns={gref.columns[0]: "stem"})
    merged = feat_an.merge(gref, on="stem", how="left")
    display(Markdown("### Anchored: ease_score vs Gurobi obj_15min (correlation)"))
    if "obj_15min" in merged.columns:
        corr = merged[["ease_score", "obj_15min"]].corr().iloc[0, 1]
        print(f"corr(ease_score, obj_15min) = {corr:.4f}  (negative expected if easier → lower cost)")
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.scatter(merged["ease_score"], merged["obj_15min"], alpha=0.7)
        ax.set_xlabel("ease_score (↑ easier)")
        ax.set_ylabel("Gurobi obj @ 15min")
        ax.set_title("Anchored Cus_100")
        plt.tight_layout()
        plt.show()

## 10. Key findings

1. **Unanchored Cus_100** has 10,000 instances (R/C/RC × wide/narrow).
2. **Easiest instances** tend to be **C + wide**, compact layouts, low battery/load pressure, wide TWs.
3. **Hardest instances** tend to be **RC + narrow** (or tight R), higher load/battery pressure, tighter windows.
4. Full ordered list (easiest→hardest) is in:
   - `analysis/cus100_unanchored_easiest_to_hardest.csv` (all families)
   - `analysis/cus100_unanchored_RC_easiest_to_hardest.csv` (RC only — matches MARL study)
5. This ranking is a **proxy**, not proven optimality; use it for curriculum / smoke tests, not as ground-truth hardness.

### Suggested use in this project
- Smoke / debug expert+BC on the **easiest 10–20 RC** first
- Keep the frozen **800/200 RC split** for the real train/test study
- Optionally curriculum: fine-tune later on harder RC tails

In [ ]:
summary = {
    "unanchored_cus100_n": int(len(feat_un)),
    "anchored_cus100_n": int(len(feat_an)),
    "rc_unanchored_n": int((feat_un.family == "RC").sum()),
    "easiest_10_all": rank.head(10)["stem"].tolist(),
    "hardest_10_all": rank.tail(10)["stem"].tolist()[::-1],
    "easiest_10_rc": rc.head(10)["stem"].tolist(),
    "hardest_10_rc": rc.tail(10)["stem"].tolist()[::-1],
    "ranking_csv": str(RANK_CSV),
    "rc_ranking_csv": str(rc_csv),
}
(OUT_DIR / "cus100_analysis_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("\nDone. Re-run with force=True in scan_cus100(...) to refresh caches.")